In [3]:
import numpy as np
import pandas as pd
import os

from google.colab import drive

In [4]:
drive_path = Path("/content/drive/MyDrive")

if not drive_path.exists():
    print("Google Drive is not mounted.")
    drive.mount("/content/drive")

project_dir = drive_path / "Nexar_Project"

print("\nGoogle Drive connected:", drive_path.exists())
print("Project directory exists:", project_dir.exists())

if project_dir.exists():
    print("\nProject folders:")

    for item in sorted(project_dir.iterdir()):
        if item.is_dir():
            print(f"[DIR]  {item.name}")
        else:
            print(f"[FILE] {item.name}")

    for folder_name in ["positive", "negative"]:

        folder = project_dir / "Dataset_2FPS" / folder_name

        print(f"\nChecking: {folder}")
        print("Directory exists:", folder.exists())

        if folder.exists():
            files = list(folder.glob("*.npz"))
            print("NPZ files found:", len(files))

else:
    print("\nNexar_Project was not found in MyDrive.")

Google Drive is not mounted.
Mounted at /content/drive

Google Drive connected: True
Project directory exists: True

Project folders:
[DIR]  Dataset_2FPS
[DIR]  Dataset_3FPS
[DIR]  Logs
[DIR]  Manifests
[DIR]  Metadata

Checking: /content/drive/MyDrive/Nexar_Project/Dataset_2FPS/positive
Directory exists: True
NPZ files found: 750

Checking: /content/drive/MyDrive/Nexar_Project/Dataset_2FPS/negative
Directory exists: True
NPZ files found: 751


In [5]:
positive_dir = Path(
    "/content/drive/MyDrive/Nexar_Project/Dataset_2FPS/positive"
)

negative_dir = Path(
    "/content/drive/MyDrive/Nexar_Project/Dataset_2FPS/negative"
)

positive_files = sorted(positive_dir.glob("*.npz"))
negative_files = sorted(negative_dir.glob("*.npz"))

print("Positive files:", len(positive_files))
print("Negative files:", len(negative_files))
print("Total files:", len(positive_files) + len(negative_files))

def inspect_npz(file_path, label):

    print(f"\n{'=' * 50}")
    print(f"{label} SAMPLE")
    print(f"File: {file_path.name}")

    try:
        with np.load(file_path, allow_pickle=False) as data:
            print("Keys:", data.files)

            for key in data.files:
                array = data[key]
                print(f"\nKey: {key}")
                print("Shape:", array.shape)
                print("Data type:", array.dtype)
                print("Size:", f"{array.nbytes / (1024**2):.2f} MB")

    except Exception as error:
        print("Error:", error)

if positive_files:
    inspect_npz(positive_files[0], "POSITIVE")

if negative_files:
    inspect_npz(negative_files[0], "NEGATIVE")

Positive files: 750
Negative files: 751
Total files: 1501

POSITIVE SAMPLE
File: 00000.npz
Keys: ['frames', 'timestamps']

Key: frames
Shape: (42, 224, 224, 3)
Data type: uint8
Size: 6.03 MB

Key: timestamps
Shape: (42,)
Data type: float32
Size: 0.00 MB

NEGATIVE SAMPLE
File: 01040.npz
Keys: ['frames', 'timestamps']

Key: frames
Shape: (79, 224, 224, 3)
Data type: uint8
Size: 11.34 MB

Key: timestamps
Shape: (79,)
Data type: float32
Size: 0.00 MB


In [8]:
positive_dir = Path(
    "/content/drive/MyDrive/Nexar_Project/Dataset_2FPS/positive"
)

negative_dir = Path(
    "/content/drive/MyDrive/Nexar_Project/Dataset_2FPS/negative"
)


def analyze_video_lengths(files, label):
    frame_counts = []
    durations = []
    timestamp_issues = []
    unreadable_files = []

    for file_path in files:

        try:
            with np.load(file_path, allow_pickle=False) as data:
                timestamps = data["timestamps"]

            frame_counts.append(len(timestamps))

            if len(timestamps) > 0:
                durations.append(
                    float(timestamps[-1] - timestamps[0])
                )

            if len(timestamps) > 1:
                if not np.all(np.diff(timestamps) > 0):
                    timestamp_issues.append(file_path.name)

        except Exception as error:
            unreadable_files.append((file_path.name, str(error)))

    if frame_counts:
        print(f"\n{'=' * 45}")
        print(f"{label} VIDEO STATISTICS")
        print(f"{'=' * 45}")

        print("Videos analyzed:", len(frame_counts))
        print("Minimum frames:", min(frame_counts))
        print("Maximum frames:", max(frame_counts))
        print("Mean frames:", round(np.mean(frame_counts), 2))
        print("Median frames:", round(np.median(frame_counts), 2))

        print("\nFrame count percentiles:")
        for p in [25, 50, 75, 90, 95, 99]:
            print(f"{p}th percentile:", round(np.percentile(frame_counts, p), 2))

        print("\nAverage timestamp duration:",
              round(np.mean(durations), 2), "seconds")

        print("Timestamp ordering issues:", len(timestamp_issues))
        print("Unreadable files:", len(unreadable_files))

        if timestamp_issues:
            print("Examples:", timestamp_issues[:5])

        if unreadable_files:
            print("Unreadable examples:", unreadable_files[:5])

    return frame_counts

positive_files = sorted(positive_dir.glob("*.npz"))
negative_files = sorted(negative_dir.glob("*.npz"))

positive_counts = analyze_video_lengths(positive_files, "POSITIVE")
negative_counts = analyze_video_lengths(negative_files, "NEGATIVE")

print("\nCLASS COMPARISON")
print("Positive videos:", len(positive_counts))
print("Negative videos:", len(negative_counts))


POSITIVE VIDEO STATISTICS
Videos analyzed: 750
Minimum frames: 7
Maximum frames: 114
Mean frames: 38.69
Median frames: 40.0

Frame count percentiles:
25th percentile: 39.0
50th percentile: 40.0
75th percentile: 41.0
90th percentile: 42.0
95th percentile: 43.0
99th percentile: 46.0

Average timestamp duration: 18.85 seconds
Timestamp ordering issues: 0
Unreadable files: 0

NEGATIVE VIDEO STATISTICS
Videos analyzed: 750
Minimum frames: 30
Maximum frames: 120
Mean frames: 74.64
Median frames: 81.0

Frame count percentiles:
25th percentile: 79.0
50th percentile: 81.0
75th percentile: 81.0
90th percentile: 82.0
95th percentile: 82.55
99th percentile: 91.0

Average timestamp duration: 36.82 seconds
Timestamp ordering issues: 0
Unreadable files: 1
Unreadable examples: [('01214_0ff59ce23a0041038f893b6b54377f5d.uploading.npz', 'File is not a zip file')]

CLASS COMPARISON
Positive videos: 750
Negative videos: 750
